# MaxText pipeline smoke test on TPU

Select an eight-device TPU runtime. This checks PP2 x TP4 and PP4 x TP2 with a
tiny four-layer, randomly initialized Qwen3-shaped model and synthetic tokens.
It checks physical parameter sharding, finite forward loss, gradients and one
SGD update. It does not measure pretrained quality or training throughput.
Install dependencies only in the TPU runtime. Each layout runs in a fresh process.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

TUNIX_REPO = "https://github.com/sontungkieu/tunix.git"
TUNIX_REVISION = "codex/maxtext-pipeline-parallelism"  # Set an exact SHA for reproducibility.
ROOT = Path("/kaggle/working/tunix") if Path("/kaggle").exists() else Path("/content/tunix")
if not ROOT.exists():
    subprocess.run(["git", "clone", TUNIX_REPO, str(ROOT)], check=True)
subprocess.run(["git", "fetch", "origin", TUNIX_REVISION], cwd=ROOT, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ROOT, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements/maxtext_requirements.txt")], check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT, check=True)


In [ ]:
# The configuration used inside the smoke script is equivalent to:
# layout = MaxTextPipelineConfig(pipeline_parallelism=2, tensor_parallelism=4,
#     num_layers_per_pipeline_stage=2, num_pipeline_microbatches=8,
#     pipeline_parallel_layers=4)
# mesh = layout.create_mesh()
# model, _ = AutoModel.from_pretrained("Qwen/Qwen3-1.7B", mesh=mesh,
#     model_source=ModelSource.MAXTEXT, maxtext_pipeline_config=layout, ...)

OUTPUT = ROOT / "smoke_results"
OUTPUT.mkdir(exist_ok=True)
env = {**os.environ, "JAX_PLATFORMS": "tpu", "PYTHONPATH": str(ROOT)}
for stages in (2, 4):
    subprocess.run([
        sys.executable, str(ROOT / "examples/model_load/maxtext_pipeline_smoke.py"),
        "--pipeline-stages", str(stages),
        "--output", str(OUTPUT / f"pp{stages}_tp{8 // stages}.json"),
    ], cwd=ROOT, env=env, check=True)
